# Exploration de l'API TED — point de départ

Objectif : comprendre la **structure réelle** des données avant de concevoir la base.

- Endpoint : `POST https://api.ted.europa.eu/v3/notices/search`
- Accès **anonyme** pour la lecture (pas de clé API nécessaire).
- Exécute les cellules une par une (Shift+Entrée) et observe ce qui revient.

> Respecte la *fair usage policy* de TED : pas de boucles massives en test.

## 1. Dépendance
Si `requests` n'est pas installé, dé-commente la ligne ci-dessous.

In [5]:
# %pip install requests
import requests, json

URL = "https://api.ted.europa.eu/v3/notices/search"

In [6]:
# On demande volontairement un champ bidon pour provoquer la liste des champs valides
body = {"query": "publication-date>=today(-3)", "fields": ["__bidon__"],
        "limit": 1, "page": 1, "scope": "ACTIVE"}
r = requests.post(URL, json=body, timeout=30)

message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = sorted(c.strip() for c in liste_brute.split(",") if c.strip())

print("Nombre total de champs :", len(champs))

# La majorité sont des codes techniques (BT-...). On garde les noms "lisibles" :
lisibles = [c for c in champs if not c.startswith("BT-") and "-" in c and c.islower()]
print("\nChamps aux noms lisibles (", len(lisibles), ") :\n")
for c in lisibles:
    print("  ", c)

Nombre total de champs : 1830

Champs aux noms lisibles ( 1326 ) :

   accessibility-justification-lot
   accessibility-lot
   acquiring-cpb-buyer
   activity-sector
   additional-classification-lot
   additional-classification-part
   additional-classification-proc
   additional-classification-type-lot
   additional-classification-type-part
   additional-classification-type-proc
   additional-info-glo
   additional-info-part
   additional-info-proc
   additional-information
   additional-information-lot
   announcement-title
   announcement-url
   assets-related-contract-extension-indicator-lot
   authority-main-activity
   award-criteria-complicated-glo
   award-criteria-order-justification
   award-criterion-complicated-lot
   award-criterion-description-glo
   award-criterion-description-lot
   award-criterion-name-glo
   award-criterion-name-lot
   award-criterion-number-fixed-glo
   award-criterion-number-fixed-lot
   award-criterion-number-lot
   award-criterion-number-threshold

## 2. Premier appel
Une petite fonction utilitaire, puis un appel simple.

In [2]:
def chercher(query, fields, limit=5, scope="ACTIVE"):
    body = {"query": query, "fields": fields, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=30)
    print("HTTP", r.status_code)
    r.raise_for_status()
    return r.json()

data = chercher(
    query='notice-title ~ "cybersecurity" SORT BY publication-date DESC',
    fields=["publication-number", "notice-title", "buyer-name", "buyer-country",
            "publication-date", "notice-type", "classification-cpv", "links"],
)

print("Clés de la réponse :", list(data.keys()))
print("Total correspondant :", data["totalNoticeCount"])
print("Annonces renvoyées :", len(data["notices"]))

HTTP 200
Clés de la réponse : ['notices', 'totalNoticeCount', 'iterationNextToken', 'timedOut']
Total correspondant : 73
Annonces renvoyées : 5


## 3. Regarder UNE annonce en détail
C'est ici que tu découvres les formats : champs multilingues, listes, dates avec fuseau, etc.

In [3]:
annonce = data["notices"][0]
print(json.dumps(annonce, indent=2, ensure_ascii=False))

{
  "notice-type": "can-standard",
  "publication-number": "412399-2026",
  "classification-cpv": [
    "72000000",
    "72000000"
  ],
  "buyer-name": {
    "eng": [
      "European Commission, DG DIGIT - Digital Services",
      "EP - European Parliament"
    ]
  },
  "publication-date": "2026-06-16+02:00",
  "links": {
    "xml": {
      "MUL": "https://ted.europa.eu/en/notice/412399-2026/xml"
    },
    "pdf": {
      "BUL": "https://ted.europa.eu/bg/notice/412399-2026/pdf",
      "SPA": "https://ted.europa.eu/es/notice/412399-2026/pdf",
      "CES": "https://ted.europa.eu/cs/notice/412399-2026/pdf",
      "DAN": "https://ted.europa.eu/da/notice/412399-2026/pdf",
      "DEU": "https://ted.europa.eu/de/notice/412399-2026/pdf",
      "EST": "https://ted.europa.eu/et/notice/412399-2026/pdf",
      "ELL": "https://ted.europa.eu/el/notice/412399-2026/pdf",
      "ENG": "https://ted.europa.eu/en/notice/412399-2026/pdf",
      "FRA": "https://ted.europa.eu/fr/notice/412399-2026/pdf",
    

## 4. Variantes de requête à tester (toutes validées)
Modifie le `query` et relance. Quelques briques qui fonctionnent :

| But | Morceau de requête |
|---|---|
| Filtrer par CPV (préfixe 72 = informatique) | `classification-cpv=72*` |
| Annonces de mise en concurrence (pas attribution) | `notice-type=cn-standard` |
| Recherche texte | `notice-title ~ "security"` |
| Pays de l'acheteur | `buyer-country=FRA` (ou `TUN`) |
| Publiées dans les 7 derniers jours | `publication-date>=today(-7)` |
| Combiner | relier avec `AND` |

Exemple combiné ci-dessous.

In [4]:
data2 = chercher(
    query='classification-cpv=72* AND notice-type=cn-standard AND publication-date>=today(-7) '
          'SORT BY publication-date DESC',
    fields=["publication-number", "notice-title", "buyer-country", "classification-cpv", "notice-type"],
    limit=5,
)
print("Total :", data2["totalNoticeCount"], "\n")
for n in data2["notices"]:
    titre = n.get("notice-title", {})
    print(n["publication-number"], "|", n.get("buyer-country"), "|", n.get("classification-cpv"))

HTTP 200
Total : 643 

411950-2026 | ['FIN'] | ['72000000', '72000000']
410415-2026 | ['DEU'] | ['79811000', '72512000', '79811000', '72512000']
410072-2026 | ['POL'] | ['48000000', '48420000', '48500000', '48517000', '48600000', '48620000', '48821000', '72000000', '72220000', '72267000', '48000000', '48420000', '48500000', '48517000', '48600000', '48620000', '48821000', '72000000', '72220000', '72267000']
410521-2026 | ['DNK'] | ['72000000', '72000000']
410566-2026 | ['FRA'] | ['72224000', '71521000', '71248000', '72224000', '71521000', '71248000']


## 5. Questions à te poser en explorant
Note tes réponses : elles serviront à concevoir la table `opportunites`.

1. Quels champs sont **toujours** présents, lesquels sont souvent absents ?
2. Quels `notice-type` veux-tu garder (écarter les avis d'attribution déjà attribués) ?
3. Comment normaliser les **dates** (format `2026-06-16+02:00`) ?
4. Comment gérer les **titres multilingues** (quelle langue de repli si pas d'anglais) ?
5. `publication-number` est-il toujours unique ? (→ clé de déduplication)

Doc de la syntaxe « expert search » : https://docs.ted.europa.eu (rechercher *expert search*).